create a bronze volume

In [0]:
%sql
use catalog youtube_dev;
use schema bronze;
create volume if not exists earthquake_data

In [0]:
dbutils.widgets.text("Catelog_name","youtube_dev","catelog")
Catelog_name=dbutils.widgets.get("Catelog_name")
print(Catelog_name)

create external connection

In [0]:
%sql
create connection if not exists earthquake_conn
type http
options (
host = 'https://earthquake.usgs.gov',
port = 443,
base_path = '/earthquakes/feed/v1.0/',
bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn=w.connections.get("earthquake_conn")
base_url= f"{conn.options['host']}{conn.options['base_path']}"
#print(base_url)

Fetch the data using API

In [0]:
import requests
import json
import datetime
response=requests.get(f"{base_url}summary/all_day.geojson")
if response.status_code != 200:
  raise Exception(f"Error: {response.status_code} - {response.text}")
current_date=datetime.datetime.now().strftime("%Y-%m-%d")
data=response.json()

Store data to volume

In [0]:
dbutils.fs.put(f"/Volumes/{Catelog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",json.dumps(data),overwrite=True)
